# ДЗ 3.3. Фракталы как вычислительный эксперимент (Зелёный + Жёлтый уровни)

## Постановка задачи

Берём комплексное отображение

$$z_{n+1} = z_n^2 + c.$$

Два родственных объекта:

- **Множество Мандельброта** $M$: фиксируем $z_0 = 0$ и варьируем параметр $c$. Точка $c$ попадает в $M$, если орбита не уходит на бесконечность.
- **Заполненное множество Джулиа** $K_c$: фиксируем $c$ и варьируем начальную точку $z_0$. Точка $z_0$ попадает в $K_c$, если её орбита ограничена.
- **Множество Джулиа** $J_c$ — это граница $K_c$.

Практическое условие «ухода»: $|z| > R$. Для Мандельброта берём $R = 2$; для Джулиа — $R = \max(2, |c|)$ (при $|c| > 2$ орбита гарантированно улетает, так что меньший радиус брать нельзя).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

plt.rcParams['figure.dpi'] = 100

## Вспомогательные функции

In [ ]:
def mandelbrot(x_range, y_range, width=600, height=600,
               max_iter=200, R=2.0):
    """
    Карта Мандельброта: z0 = 0, c пробегает сетку.
    Возвращает массив номеров итераций выхода.
    """
    x = np.linspace(*x_range, width)
    y = np.linspace(*y_range, height)
    c = x[None, :] + 1j * y[:, None]

    escape_steps = np.zeros(c.shape, dtype=int)
    z = np.zeros(c.shape, dtype=complex)
    active = np.ones(c.shape, dtype=bool)

    for k in range(1, max_iter + 1):
        z[active] = z[active]**2 + c[active]
        escaped = active & (np.abs(z) > R)
        escape_steps[escaped] = k
        active[escaped] = False
        if not active.any():
            break

    return escape_steps

def escape_time(c, x_range, y_range, width=600, height=600,
                max_iter=200, R=2.0):
    """
    Номер первой итерации выхода для z -> z^2 + c при фиксированном c.
    Возвращает (escape_steps, z_final): номер итерации и z в момент выхода.
    """
    x = np.linspace(*x_range, width)
    y = np.linspace(*y_range, height)
    z = x[None, :] + 1j * y[:, None]

    escape_steps = np.zeros(z.shape, dtype=int)
    z_final = np.zeros(z.shape, dtype=complex)
    active = np.ones(z.shape, dtype=bool)

    for k in range(1, max_iter + 1):
        z[active] = z[active]**2 + c
        escaped = active & (np.abs(z) > R)
        escape_steps[escaped] = k
        z_final[escaped] = z[escaped]
        active[escaped] = False
        if not active.any():
            break

    z_final[active] = z[active]
    return escape_steps, z_final

## Зелёный уровень. Множество Мандельброта

In [ ]:
m_xrange = (-2.0, 1.0)
m_yrange = (-1.5, 1.5)

m_escape = mandelbrot(m_xrange, m_yrange, width=600, height=600,
                      max_iter=200, R=2.0)

c_values = [
    (complex(0.0, 0.0),         'c = 0'),
    (complex(-1.0, 0.0),        'c = -1'),
    (complex(-0.123, 0.745),    'c = -0.123 + 0.745i'),
    (complex(1.0, 0.0),         'c = 1'),
]

fig, ax = plt.subplots(figsize=(12, 12))
im = ax.imshow(m_escape, extent=[*m_xrange, *m_yrange], origin='lower',
               cmap='twilight_shifted', vmin=0, vmax=200,
               interpolation='nearest')

for c, label in c_values:
    ax.plot(c.real, c.imag, 'o', color='red', markersize=8,
            markeredgecolor='white', markeredgewidth=1.5)
    dx = -70 if abs(c.real - 1.0) < 1e-6 else 8
    ax.annotate(label, (c.real, c.imag), textcoords='offset points',
                xytext=(dx, 8), color='white', fontsize=9,
                bbox=dict(boxstyle='round', fc='black', alpha=0.6))

ax.set_xlabel('Re(c)')
ax.set_ylabel('Im(c)')
ax.set_title('Множество Мандельброта, $z_{n+1} = z_n^2 + c$\n'
             'сетка 600×600, $N_{max} = 200$, $R = 2$')
ax.set_aspect('equal')
fig.colorbar(im, ax=ax, pad=0.02,
             label='Номер итерации выхода (0—200)\n'
                   '0 — не вышла за 200 шагов')
plt.tight_layout()
plt.show()

# Четыре множества Джулиа

Теперь фиксируем $c$ и строим карту по $z_0$. Радиус выхода для Джулиа берём $R = \max(2, |c|)$.

In [ ]:
def plot_julia(c, x_range=(-2, 2), y_range=(-2, 2),
               width=600, height=600, max_iter=200):
    R = max(2.0, abs(c))
    esc, _ = escape_time(c, x_range, y_range,
                         width=width, height=height,
                         max_iter=max_iter, R=R)

    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.imshow(esc, extent=[*x_range, *y_range], origin='lower',
                   cmap='turbo', vmin=0, vmax=max_iter,
                   interpolation='nearest')
    ax.set_title(f'Заполненное множество Джулиа, $c = {c}$\n'
                 f'сетка {width}×{height}, $N_{{max}} = {max_iter}$, '
                 f'$R = \\max(2, |c|) = {R:.3f}$')
    ax.set_xlabel('Re($z_0$)')
    ax.set_ylabel('Im($z_0$)')
    ax.set_aspect('equal')
    fig.colorbar(im, ax=ax, pad=0.02,
                 label=f'Номер итерации выхода (0—{max_iter})\n'
                       f'0 — не вышла за {max_iter} шагов')
    plt.tight_layout()
    plt.show()
    return esc

for c, label in c_values:
    plot_julia(c)

## Контрольный случай $c = 0$

При $c = 0$ отображение упрощается до $z_{n+1} = z_n^2$, и ответ известен аналитически: орбита ограничена ровно при $|z_0| \le 1$. То есть $K_0$ — замкнутый единичный диск, $J_0$ — единичная окружность.

In [ ]:
esc0, _ = escape_time(complex(0.0), (-2, 2), (-2, 2),
                      width=600, height=600, max_iter=200, R=2.0)

x = np.linspace(-2, 2, 600)
y = np.linspace(-2, 2, 600)
Z = x[None, :] + 1j * y[:, None]
inside_disk = np.abs(Z) <= 1.0

algorithm_inside = (esc0 == 0)
mismatch = algorithm_inside != inside_disk
print(f"Несовпадений: {mismatch.sum()} из {mismatch.size} "
      f"({100 * mismatch.mean():.4f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(esc0, extent=[-2, 2, -2, 2], origin='lower',
                     cmap='turbo', vmin=0, vmax=200,
                     interpolation='nearest')
axes[0].set_title('Алгоритм: номер итерации выхода\n'
                  'сетка 600×600, $N_{max} = 200$, $R = 2$\n'
                  '0 — не вышла за 200 шагов')
axes[0].set_xlabel('Re($z_0$)')
axes[0].set_ylabel('Im($z_0$)')
axes[0].set_aspect('equal')
fig.colorbar(im0, ax=axes[0], pad=0.02,
             label='Номер итерации выхода')

axes[1].imshow(inside_disk, extent=[-2, 2, -2, 2], origin='lower',
               cmap='gray_r')
axes[1].set_title('Аналитика: $|z_0| \\leq 1$ — единичный диск')
axes[1].set_xlabel('Re($z_0$)')
axes[1].set_ylabel('Im($z_0$)')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

### Что можно утверждать о точке, не вышедшей за $N_{\max}$ шагов?

Строго говоря — почти ничего. То, что за 200 итераций орбита не покинула круг радиуса $R$, не доказывает её ограниченности: точка может выйти на 201-й итерации.

Что **можно** утверждать: если точка вышла, она точно не принадлежит множеству. Обратное неверно.

## Жёлтый уровень. Увеличение фрагмента границы

Возьмём маленький кусок границы Мандельброта: окрестность точки «морского конька» вблизи $c \approx -0.75 + 0.1i$. Нас интересует, как меняется картинка при

- увеличении разрешения с $600\times 600$ до $1200\times 1200$ (при том же $N_{\max}$);
- увеличении $N_{\max}$ с 200 до 800 (при том же разрешении).

In [ ]:
combos = [
    # width, max_iter, vmax, title
    (600,  200, 200, '600×600, $N_{max}=200$'),
    (1200, 200, 200, '1200×1200, $N_{max}=200$'),
    (600,  800, 200, '600×600, $N_{max}=800$'),
    (1200, 800, 200, '1200×1200, $N_{max}=800$'),
]

zoom_x = (-1.77, -1.74)
zoom_y = (0.005, 0.035)

fig, axes = plt.subplots(2, 2, figsize=(13, 13))
for ax, (w, nmax, vmax, title) in zip(axes.ravel(), combos):
    esc = mandelbrot(zoom_x, zoom_y, width=w, height=w,
                     max_iter=nmax, R=2.0)
    im = ax.imshow(esc, extent=[*zoom_x, *zoom_y], origin='lower',
                   cmap='twilight_shifted', vmin=0, vmax=vmax,
                   interpolation='nearest')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Re(c)')
    ax.set_ylabel('Im(c)')
    ax.set_aspect('equal')
    fig.colorbar(im, ax=ax, fraction=0.046,
                 label='Номер итерации выхода (общая шкала 0—200)')

plt.suptitle('Влияние разрешения и $N_{max}$ на фрагмент границы', fontsize=13, y=0.995)
plt.tight_layout()
plt.show()

## Плавная окраска

Целая окраска «номер итерации выхода» даёт ступеньки: соседние пиксели, вышедшие на одной и той же итерации, одинаковы по цвету, и на границе видны кольца.

Идея плавной окраски: вместо $k$ брать непрерывную величину

$$\mu = k - \frac{\ln \ln |z_k|}{\ln 2},$$

где $z_k$ — значение орбиты в момент выхода. Откуда формула: при больших $|z|$ отображение даёт $|z_{n+1}| \approx |z_n|^2$, то есть $\ln|z_{n+1}| \approx 2\ln|z_n|$, и $\ln\ln|z|$ растёт линейно со скоростью $\ln 2$ за итерацию. Вычитая нормированное $\ln\ln|z_k|$, мы компенсируем эту «лишнюю» часть и получаем величину, которая меняется непрерывно при переходе через границу.

Для не вышедших точек полагаем $\mu = 0$ — они идут отдельным цветом.

In [ ]:
def smooth_mandelbrot(x_range, y_range, width=800, height=800,
                      max_iter=400, R=2.0):
    """
    Плавная оценка номера выхода для множества Мандельброта:
    mu = k - ln ln|z_k| / ln 2,  z0 = 0, c пробегает сетку.
    """
    x = np.linspace(*x_range, width)
    y = np.linspace(*y_range, height)
    c = x[None, :] + 1j * y[:, None]

    mu = np.zeros(c.shape, dtype=float)
    z = np.zeros(c.shape, dtype=complex)
    active = np.ones(c.shape, dtype=bool)

    for k in range(1, max_iter + 1):
        z[active] = z[active]**2 + c[active]
        escaped = active & (np.abs(z) > R)
        if escaped.any():
            zk = z[escaped]
            mu[escaped] = k - np.log(np.log(np.abs(zk))) / np.log(2)
        active[escaped] = False
        if not active.any():
            break

    return mu


frac_x = (-0.76, -0.73)
frac_y = (0.08, 0.11)
W = H = 800
NMAX = 400
VM = 100

esc_int    = mandelbrot(frac_x, frac_y, width=W, height=H,
                        max_iter=NMAX, R=2.0)
esc_smooth = smooth_mandelbrot(frac_x, frac_y, width=W, height=H,
                               max_iter=NMAX, R=2.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

im0 = axes[0].imshow(esc_int, extent=[*frac_x, *frac_y], origin='lower',
                     cmap='twilight_shifted', vmin=0, vmax=VM,
                     interpolation='nearest')
axes[0].set_title('Целая окраска: номер итерации $k$\n'
                  f'сетка {W}×{H}, $N_{{max}}={NMAX}$, $R=2$')
axes[0].set_xlabel('Re(c)')
axes[0].set_ylabel('Im(c)')
axes[0].set_aspect('equal')
fig.colorbar(im0, ax=axes[0], pad=0.02,
             label=f'$k$ (0 — не вышла, $\\geq${VM} — верх шкалы)')

im1 = axes[1].imshow(esc_smooth, extent=[*frac_x, *frac_y], origin='lower',
                     cmap='twilight_shifted', vmin=0, vmax=VM,
                     interpolation='nearest')
axes[1].set_title('Плавная: $\\mu = k - \\ln\\ln|z_k|/\\ln 2$\n'
                  f'сетка {W}×{H}, $N_{{max}}={NMAX}$, $R=2$')
axes[1].set_xlabel('Re(c)')
axes[1].set_ylabel('Im(c)')
axes[1].set_aspect('equal')
fig.colorbar(im1, ax=axes[1], pad=0.02,
             label=f'$\\mu$ — плавный номер итерации\n')

plt.tight_layout()
plt.show()

## Анимация: $c$ идёт по прямой через границу $M$

Возьмём вертикальную прямую $c(t) = -0.75 + t \cdot i$, $t \in [-0.2, 0.2]$. Она проходит через «морского конька» и пересекает границу Мандельброта.

Слева — карта $M$ с движущейся красной точкой. Справа — множество Джулиа для текущего $c$, а поверх него — критическая орбита $z_0 = 0$ (первые ~120 точек). Смысл: пока $c$ внутри $M$, критическая орбита ограничена и Джулиа связно; как только $c$ выходит за границу, критическая орбита улетает и Джулиа рассыпается. Это и есть теорема о связности, но увиденная глазами, а не выведенная.

In [ ]:
t_vals = np.linspace(-0.2, 0.4, 80)
c_curve = -0.5 + 1j * t_vals

julia_frames = []
for c in c_curve:
    R = max(2.0, abs(c))
    esc, _ = escape_time(c, (-1.5, 1.5), (-1.5, 1.5),
                         width=300, height=300, max_iter=120, R=R)
    julia_frames.append(esc)
julia_frames = np.array(julia_frames)


def critical_orbit(c, n=120):
    """Орбита z0 = 0 для отображения z -> z^2 + c."""
    z = 0.0 + 0.0j
    orbit = [z]
    for _ in range(n):
        z = z**2 + c
        orbit.append(z)
        if abs(z) > 10:
            break
    return np.array(orbit)


crit_orbits = [critical_orbit(c) for c in c_curve]

fig, (ax_map, ax_julia) = plt.subplots(1, 2, figsize=(13, 6))

m_esc = mandelbrot((-2, 1), (-1.5, 1.5), width=400, height=400,
                   max_iter=200, R=2.0)
ax_map.imshow(m_esc, extent=[-2, 1, -1.5, 1.5], origin='lower',
              cmap='twilight_shifted', vmin=0, vmax=200)
ax_map.plot(c_curve.real, c_curve.imag, 'r-', lw=1.5, alpha=0.7)
point_c, = ax_map.plot([], [], 'ro', markersize=10)
ax_map.set_xlim(-2, 1)
ax_map.set_ylim(-1.5, 1.5)
ax_map.set_aspect('equal')
ax_map.set_title('Параметр $c$ на карте Мандельброта')
ax_map.set_xlabel('Re(c)')
ax_map.set_ylabel('Im(c)')

im_julia = ax_julia.imshow(julia_frames[0],
                           extent=[-1.5, 1.5, -1.5, 1.5], origin='lower',
                           cmap='twilight_shifted',
                           vmin=0, vmax=120, animated=True)
orbit_line, = ax_julia.plot([], [], 'r.', markersize=4)
ax_julia.set_title('Множество Джулиа')
ax_julia.set_xlabel('Re($z_0$)')
ax_julia.set_ylabel('Im($z_0$)')
ax_julia.set_aspect('equal')


def update(frame):
    im_julia.set_array(julia_frames[frame])
    point_c.set_data([c_curve[frame].real], [c_curve[frame].imag])
    orb = crit_orbits[frame]
    orbit_line.set_data(orb.real, orb.imag)
    ax_julia.set_title(f'Множество Джулиа, $c = {c_curve[frame]:.3f}$')
    return im_julia, point_c, orbit_line


anim = animation.FuncAnimation(fig, update, frames=len(c_curve),
                               interval=150, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

### Связь критической орбиты и связности $J_c$

Критическая орбита — это орбита точки $z_0 = 0$. Для отображения $z \mapsto z^2 + c$ она играет особую роль: **множество Джулиа $J_c$ связно тогда и только тогда, когда критическая орбита ограничена**. Это классическая теорема (Джулиа, Фату). На практике «ограничена» означает «не уходит на бесконечность», и это в точности условие $c \in M$.

В анимации это видно напрямую: пока красная точка на левой панели внутри $M$, критические орбиты на правой панели остаются в ограниченной области, а само множество Джулиа — одна связная «клякса». Как только $c$ выходит за границу $M$, орбита начинает улетать, и Джулиа распадается на несвязную «пыль Кантора».

**Ограничения конечного разрешения.**

1. Тонкие нити границы, шириной меньше пикселя, теряются или сливаются с фоном.
2. Мелкие компоненты связности (размером в несколько пикселей) могут быть не видны, и множество выглядит «более связным», чем оно есть.
3. Точки, вышедшие на итерации $N_{\max}+1$, формально уже вне $K_c$, но на картинке покрашены как «не вышла». Это делает границу чуть шире реальной.

Поэтому визуальная связность на экране — не доказательство. Доказательство — теорема о критической орбите; картинка лишь иллюстрирует её.